# 08 · TinyStories: o modelo base da Fase 2

No Shakespeare, um GPT de 10M parâmetros tem ~1M caracteres para aprender e acaba **decorando**. O [TinyStories](https://arxiv.org/abs/2305.07759) (Eldan e Li, 2023) é o oposto: milhões de histórias curtas escritas pelo GPT-4 com o vocabulário de uma criança de 3 a 4 anos. O artigo mostra que modelos de poucos milhões de parâmetros já escrevem histórias coerentes nesse domínio.

Aqui usamos os primeiros 50 MB do arquivo de treino (o inteiro tem 2,2 GB) e o arquivo de validação inteiro, ainda com **tokenização por caractere**. Tokenização BPE é o experimento `F2-TOK` da Fase 2.

**Entrega da Fase 1:** um GPT de ~10M parâmetros, escrito e entendido por você, gerando histórias coerentes. Diário: `F1-TS`.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import torch

from lab.data import EOT, get_batch, tinystories
from lab.model import GPT, GPTConfig, count_params
from lab.plot import compare, table
from lab.train import generate_text, load_checkpoint, pick_device, train

DEVICE = pick_device()
ds = tinystories(max_mb=50)  # na primeira vez baixa ~70 MB do Hugging Face para data/
tok, V = ds.tok, ds.tok.vocab_size
print(ds, "| dispositivo:", DEVICE)
print("vocabulário:", repr("".join(tok.chars)))

In [ ]:
story = tok.decode(ds.val[:3000].tolist()).split(EOT)[0]
print(story)

## O modelo de ~10M parâmetros

Mesma arquitetura do notebook 07. Sem dropout: com 50 MB e 5000 passos o modelo vê cada caractere menos de uma vez, então não há o que decorar.

In [ ]:
CFG = GPTConfig(vocab_size=V, block_size=256, d_model=384, n_layer=6, n_head=6, attn="batched", dropout=0.0)
TRAIN = dict(steps=5000, lr=6e-4, batch_size=32, grad_clip=1.0, eval_every=250, device=DEVICE,
             sample_prompt="Once upon a time", sample_tokens=500, save=True)
torch.manual_seed(1337)
print(f"{count_params(GPT(CFG)):,} parâmetros")

### Quanto vai demorar?

In [ ]:
def sync():
    if DEVICE == "mps":
        torch.mps.synchronize()

torch.manual_seed(1337)
m = GPT(CFG).to(DEVICE)
opt = torch.optim.AdamW(m.parameters(), lr=TRAIN["lr"])
for i in range(15):
    if i == 5:
        sync()
        t0 = time.time()
    _, loss = m(*get_batch(ds.train, TRAIN["batch_size"], CFG.block_size, DEVICE))
    opt.zero_grad()
    loss.backward()
    opt.step()
sync()
per_step = (time.time() - t0) / 10
print(f"{per_step * 1000:.0f} ms por passo -> {TRAIN['steps']} passos em ~{per_step * TRAIN['steps'] / 60:.0f} min")
del m, opt

## O treino

**Minha previsão:** _(loss de validação final? em que passo o texto começa a ter frases inteiras?)_

Se a estimativa acima passar de ~30 min, rode no terminal (o resultado vai para o mesmo lugar e as células seguintes funcionam igual):

```bash
python -m lab.train --name 08-tinystories-10m --dataset tinystories --max_mb 50 \
    --n_layer 6 --n_head 6 --d_model 384 --block_size 256 --attn batched \
    --steps 5000 --lr 6e-4 --batch_size 32 --grad_clip 1.0 --eval_every 250 \
    --sample_prompt "Once upon a time" --sample_tokens 500 --save
```

In [ ]:
torch.manual_seed(1337)
run = train(GPT(CFG), ds, name="08-tinystories-10m", **TRAIN)
print(run.sample)

In [ ]:
compare(["08-tinystories-10m"], title="GPT 10M no TinyStories")
table(["07-medio-drop0.2", "08-tinystories-10m"])

## Histórias
A geração para quando o modelo emite o caractere de fim de história.

In [ ]:
model, tok = load_checkpoint("08-tinystories-10m", DEVICE)
stop = tok.stoi[EOT]
for prompt in ["Once upon a time", "Tom and his dog", "The little girl was sad because", "One day, a big"]:
    print("---")
    print(generate_text(model, tok, prompt, 800, temperature=0.8, stop_token=stop).replace(EOT, "\n[fim]"))

---
## Experimentos

### 1. Quantidade de dados
Um modelo pequeno (~0,8M parâmetros, o tamanho da Fase 0), 3000 passos, treinado com 1 MB e com 50 MB de histórias.

**Minha previsão:** _(com 1 MB aparece overfitting como no Shakespeare? quanto de diferença na validação?)_

In [ ]:
def pequeno(vocab_size):
    torch.manual_seed(1337)
    return GPT(GPTConfig(vocab_size=vocab_size, block_size=128, d_model=128, n_layer=4, n_head=4, attn="batched"))

SMALL = dict(steps=3000, lr=1e-3, batch_size=32, eval_every=250, device=DEVICE, sample_prompt="Once upon a time", verbose=False)
ds1 = tinystories(max_mb=1)
print(ds1)
print(train(pequeno(ds1.tok.vocab_size), ds1, name="08-pequeno-1mb", **SMALL))
print(train(pequeno(V), ds, name="08-pequeno-50mb", **SMALL))
compare(prefix="08-pequeno", title="Mesmo modelo, 1 MB vs. 50 MB")
table(prefix="08-pequeno")

### 2. Tamanho do modelo com os mesmos dados
**Minha previsão:** _(quanto o modelo de 10M ganha sobre o pequeno de 0,8M, por passo e por segundo?)_

In [ ]:
compare(["08-pequeno-50mb", "08-tinystories-10m"], title="0,8M vs. 10M (por passo)")
compare(["08-pequeno-50mb", "08-tinystories-10m"], x="elapsed_s", title="0,8M vs. 10M (por segundo)")
table(["08-pequeno-50mb", "08-tinystories-10m"])

---
## Entrega da Fase 1

Marque no `PLANO.md` quando conseguir responder "sim" a tudo:

- [ ] Consigo explicar cada linha de `lab/model.py` sem consultar nada.
- [ ] O GPT de ~10M gera histórias com começo, meio e fim, com personagens consistentes.
- [ ] Registrei no diário uma entrada para cada notebook (F1-BIG, F1-ATN, F1-MHA, F1-MLP, F1-RES, F1-NRM, F1-SHK, F1-TS).
- [ ] Sei dizer o ganho de cada peça nos meus experimentos.

O checkpoint `runs/08-tinystories-10m/model.pt` é o **modelo base** da Fase 2: os experimentos de arquitetura vão comparar variações contra ele.

## Para o diário (`F1-TS-01`)

- Loss de validação final e uma história gerada de que você gostou e uma que deu errado. O que deu errado?
- O que muda entre 1 MB e 50 MB? Relacione com o overfitting do notebook 07.
- Se você tivesse mais 1 hora de GPU, gastaria em mais passos, mais dados ou modelo maior? Por quê?